# Experiment 2: Adding Tool Calls
### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

---

In Stage 1 we built a bare completions agent and learned MLflow's evaluation primitives. The agent could answer cooking questions from its training data, but it couldn't *do* anything — no recipe lookup, no nutrition data, no real-world information.

In this stage we give the agent **tools** — functions it can call to look up recipes and get nutrition information. This is the first real step toward an *agent* rather than a chatbot.

More importantly, we'll see how MLflow handles tool-calling agents:

- **Automatic span capture** — tool calls appear as `TOOL` spans in the trace alongside the LLM spans
- **`ToolCallCorrectness`** — did the agent pick the right tool with the right arguments?
- **`ToolCallEfficiency`** — did the agent make redundant or unnecessary tool calls?
- **`expected_tool_calls`** — ground truth for *which tools should have been called*

> **Prerequisites:** Complete Stage 1 or be familiar with `mlflow.genai.evaluate()`, `predict_fn`, and the scorer pattern.

---
## 0 — Environment Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import mlflow
from mlflow.entities import SpanType

load_dotenv()

print(f"MLflow version: {mlflow.__version__}")
print(f"GEMINI_API_KEY present: {os.getenv('GEMINI_API_KEY') is not None}")

In [ ]:
client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-2.5-flash-lite"

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("recipe-assistant-tools")
mlflow.openai.autolog()

print("Autologging enabled — tool call spans will appear automatically.")

---
## 1 — Defining the Tools

We'll give our cooking assistant two tools:

| Tool | What it does | When the agent should use it |
|---|---|---|
| `search_recipes` | Looks up recipes by name or ingredient | When the user asks for a specific recipe or wants ideas |
| `get_nutrition_info` | Returns nutrition facts for a food item | When the user asks about calories, macros, or dietary content |

For this workshop, the tools return **simulated data** — in production these would hit a recipe API and a nutrition database. The point is to learn the tool-calling *pattern* and how MLflow traces and evaluates it.

### Making tools visible to MLflow

The key decorator is `@mlflow.trace(span_type=SpanType.TOOL)`. This tells MLflow to record each tool invocation as a `TOOL` span in the trace — capturing inputs, outputs, and timing. Without this, the tool calls would be invisible in the trace.

```python
@mlflow.trace(span_type=SpanType.TOOL)
def my_tool(arg: str) -> dict:
    ...
```

In [ ]:
# ── Simulated recipe database ──────────────────────────────────────────────
RECIPE_DB = {
    "pancakes": {
        "name": "Classic Pancakes",
        "prep_time": "5 minutes",
        "cook_time": "15 minutes",
        "servings": 4,
        "ingredients": [
            "1.5 cups all-purpose flour",
            "3.5 tsp baking powder",
            "1 tbsp sugar",
            "1/4 tsp salt",
            "1.25 cups milk",
            "1 egg",
            "3 tbsp melted butter",
        ],
        "instructions": [
            "Mix dry ingredients in a large bowl.",
            "Make a well in the center, pour in milk, egg, and melted butter.",
            "Mix until smooth.",
            "Heat a griddle over medium-high heat and grease lightly.",
            "Pour batter onto griddle using about 1/4 cup per pancake.",
            "Cook until bubbles form on surface, then flip and cook until golden.",
        ],
    },
    "pasta_carbonara": {
        "name": "Pasta Carbonara",
        "prep_time": "10 minutes",
        "cook_time": "20 minutes",
        "servings": 4,
        "ingredients": [
            "400g spaghetti",
            "200g guanciale or pancetta, diced",
            "4 large egg yolks",
            "1 cup Pecorino Romano, finely grated",
            "Freshly cracked black pepper",
        ],
        "instructions": [
            "Cook spaghetti in well-salted boiling water until al dente. Reserve 1 cup pasta water.",
            "While pasta cooks, crisp guanciale in a cold pan over medium heat for 8-10 minutes.",
            "Whisk egg yolks and Pecorino together in a bowl.",
            "Drain pasta and toss with guanciale off heat.",
            "Add egg mixture and toss vigorously, adding pasta water a splash at a time until creamy.",
            "Serve immediately with extra Pecorino and black pepper.",
        ],
    },
    "chicken_stir_fry": {
        "name": "Chicken Stir Fry",
        "prep_time": "15 minutes",
        "cook_time": "10 minutes",
        "servings": 4,
        "ingredients": [
            "500g chicken breast, sliced thin",
            "2 tbsp soy sauce",
            "1 tbsp sesame oil",
            "2 cups mixed vegetables (bell pepper, broccoli, snap peas)",
            "3 cloves garlic, minced",
            "1 tbsp fresh ginger, grated",
            "2 tbsp vegetable oil",
            "1 tbsp cornstarch",
        ],
        "instructions": [
            "Toss chicken with soy sauce and cornstarch. Let sit 5 minutes.",
            "Heat vegetable oil in a wok over high heat until smoking.",
            "Stir-fry chicken in batches until golden, about 3 minutes. Set aside.",
            "Add sesame oil, garlic, and ginger to wok. Cook 30 seconds.",
            "Add vegetables and stir-fry 3-4 minutes until crisp-tender.",
            "Return chicken to wok, toss to combine, and serve over rice.",
        ],
    },
}

# ── Simulated nutrition database ───────────────────────────────────────────
NUTRITION_DB = {
    "chicken breast": {"calories": 165, "protein": 31, "carbs": 0, "fat": 3.6, "per": "100g"},
    "spaghetti": {"calories": 158, "protein": 5.8, "carbs": 31, "fat": 0.9, "per": "100g cooked"},
    "egg": {"calories": 72, "protein": 6.3, "carbs": 0.4, "fat": 4.8, "per": "1 large"},
    "avocado": {"calories": 240, "protein": 3, "carbs": 13, "fat": 22, "per": "1 medium"},
    "pancake": {"calories": 86, "protein": 2.4, "carbs": 11, "fat": 3.5, "per": "1 medium (6 inch)"},
    "guanciale": {"calories": 655, "protein": 8.9, "carbs": 0, "fat": 69, "per": "100g"},
}

print(f"Recipe DB: {list(RECIPE_DB.keys())}")
print(f"Nutrition DB: {list(NUTRITION_DB.keys())}")

In [ ]:
@mlflow.trace(span_type=SpanType.TOOL)
def search_recipes(query: str) -> str:
    """Search the recipe database by name or keyword."""
    query_lower = query.lower()
    matches = []
    for key, recipe in RECIPE_DB.items():
        # Match on recipe key, name, or any ingredient
        if (query_lower in key
            or query_lower in recipe["name"].lower()
            or any(query_lower in ing.lower() for ing in recipe["ingredients"])):
            matches.append(recipe)

    if not matches:
        return json.dumps({"found": False, "message": f"No recipes found for '{query}'"})

    return json.dumps({"found": True, "recipes": matches}, indent=2)


@mlflow.trace(span_type=SpanType.TOOL)
def get_nutrition_info(food_item: str) -> str:
    """Get nutrition facts for a food item."""
    food_lower = food_item.lower()
    # Try exact match first, then partial match
    for key, info in NUTRITION_DB.items():
        if food_lower in key or key in food_lower:
            return json.dumps({"found": True, "item": key, **info})

    return json.dumps({"found": False, "message": f"No nutrition data for '{food_item}'"})


# Quick test
print(search_recipes("pancakes")[:200], "...")
print()
print(get_nutrition_info("chicken breast"))

---
## 2 — The Tool Schema (OpenAI Function Calling Format)

The OpenAI SDK uses a specific JSON schema to describe tools to the LLM. The model reads these descriptions and decides *when* and *how* to call each tool. Good descriptions are critical — a vague description leads to wrong tool selection.

This schema works identically whether you're pointing the SDK at OpenAI, Gemini, or any compatible endpoint.

In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_recipes",
            "description": (
                "Search the recipe database for recipes by name, dish type, or ingredient. "
                "Returns matching recipes with ingredients, instructions, and timing. "
                "Use this when the user asks for a specific recipe, wants recipe ideas, "
                "or asks what to cook with certain ingredients."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The recipe name, dish type, or ingredient to search for",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_nutrition_info",
            "description": (
                "Get nutrition facts (calories, protein, carbs, fat) for a specific food item. "
                "Use this when the user asks about calories, macros, nutrition, "
                "or dietary content of a food."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "food_item": {
                        "type": "string",
                        "description": "The food item to look up (e.g. 'chicken breast', 'egg')",
                    },
                },
                "required": ["food_item"],
            },
        },
    },
]

print("Tools registered:")
for t in tools_schema:
    print(f"  - {t['function']['name']}: {t['function']['description'][:60]}...")

---
## 3 — Building the Tool-Calling Agent

A tool-calling agent follows a loop:

1. **Send** the user message + tool schemas to the LLM
2. **Check** if the LLM response contains tool calls
3. **Execute** each tool call and append the results to the conversation
4. **Send again** so the LLM can incorporate the tool results into its final answer
5. **Repeat** until the LLM responds with text (no more tool calls)

We wrap the whole loop in `@mlflow.trace` so the entire agent execution — LLM calls, tool calls, and all — appears as a single trace with nested spans.

In [ ]:
SYSTEM_PROMPT = """You are a helpful cooking assistant for Mise en Place, a cooking app.
Answer questions about recipes, cooking techniques, ingredients, and meal planning.

You have access to tools:
- search_recipes: Look up recipes from our database. Use this when the user asks for a recipe.
- get_nutrition_info: Get nutrition facts. Use this when the user asks about calories or nutrition.

Guidelines:
- Always include specific measurements and time estimates
- Be conversational and encouraging, like a knowledgeable friend
- Use your tools when relevant — don't make up recipe details when you can look them up
- For general cooking technique questions, answer from your knowledge (no tool needed)"""


# Map tool names to their Python functions
TOOL_DISPATCH = {
    "search_recipes": search_recipes,
    "get_nutrition_info": get_nutrition_info,
}


@mlflow.trace
def recipe_agent(inputs: dict) -> str:
    """Tool-calling recipe assistant."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": inputs["question"]},
    ]

    # Agent loop — keep going until the model responds with text
    for _ in range(5):  # safety limit to prevent infinite loops
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_schema,
            temperature=0.1,
            max_completion_tokens=1024,
        )

        message = response.choices[0].message

        # If no tool calls, we have our final answer
        if not message.tool_calls:
            return message.content

        # Process each tool call
        messages.append(message)  # append the assistant's tool-call message

        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            # Execute the tool
            tool_fn = TOOL_DISPATCH.get(fn_name)
            if tool_fn:
                result = tool_fn(**fn_args)
            else:
                result = json.dumps({"error": f"Unknown tool: {fn_name}"})

            # Append tool result to conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return "I wasn't able to complete your request. Please try again."


print("Agent defined. Let's test it.")

---
## 4 — Testing the Agent

Let's run a few queries to see the agent in action. After each call, check the MLflow UI — you'll see traces with nested spans showing the LLM call, the tool calls, and the follow-up LLM call.

In [ ]:
# Should trigger search_recipes
result = recipe_agent({"question": "How do I make pancakes?"})
print(result)

In [ ]:
# Should trigger get_nutrition_info
result = recipe_agent({"question": "How many calories are in an avocado?"})
print(result)

In [ ]:
# General technique question — should NOT call any tools
result = recipe_agent({"question": "What's the difference between baking and roasting?"})
print(result)

In [ ]:
# Should trigger both search_recipes AND get_nutrition_info
result = recipe_agent({"question": "Give me a chicken stir fry recipe and tell me the calories in chicken breast."})
print(result)

### What to look for in the MLflow UI

Open the MLflow UI and click on the traces for the calls above. You should see:

```
recipe_agent                          ← root span (CHAIN)
  ├── chat.completions.create         ← first LLM call (CHAT_MODEL)
  ├── search_recipes                  ← tool execution (TOOL)
  └── chat.completions.create         ← follow-up LLM call with tool results (CHAT_MODEL)
```

For the multi-tool query, you'll see multiple `TOOL` spans. This nested structure is what `ToolCallCorrectness` and `ToolCallEfficiency` analyze.

---
## 5 — Evaluating Tool Call Quality

In Stage 1 we evaluated *what the agent said*. Now we also need to evaluate *what the agent did* — which tools it called and whether those calls were appropriate.

MLflow provides two specialized scorers for this:

| Scorer | What it checks | Ground truth needed? |
|---|---|---|
| `ToolCallCorrectness` | Did the agent call the right tools with appropriate arguments? | Optional — works with or without `expected_tool_calls` |
| `ToolCallEfficiency` | Did the agent make redundant or unnecessary calls? | No |

### How they work

Both scorers read the **trace** generated by your `predict_fn`, not just the text output. They look for spans with `span_type=TOOL` and evaluate the full trajectory of the agent's actions.

This is why we decorated our tools with `@mlflow.trace(span_type=SpanType.TOOL)` — without that, these scorers would have nothing to evaluate.

---
## 6 — Ground-Truth-Free Evaluation

The simplest way to use `ToolCallCorrectness` is without ground truth. The scorer uses an LLM judge to assess whether the tool calls *make sense* given the user's question and the available tools. No `expected_tool_calls` needed.

This is useful for:
- Quick sanity checks during development
- Catching obviously wrong tool selections
- Getting started before you've built a labeled eval set

In [ ]:
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency

eval_dataset_simple = [
    {"inputs": {"question": "How do I make pancakes?"}},
    {"inputs": {"question": "How many calories in an egg?"}},
    {"inputs": {"question": "What's the difference between baking and roasting?"}},
    {"inputs": {"question": "Show me a pasta carbonara recipe and how much fat is in guanciale."}},
]

results_no_gt = mlflow.genai.evaluate(
    data=eval_dataset_simple,
    predict_fn=recipe_agent,
    scorers=[
        ToolCallCorrectness(),
        ToolCallEfficiency(),
    ],
)

print("=== Ground-truth-free tool call evaluation ===")
print(results_no_gt.metrics)

In [ ]:
results_no_gt.tables["eval_results_table"]

---
## 7 — Adding `expected_tool_calls` Ground Truth

Ground-truth-free evaluation catches obvious mistakes but can't verify specific arguments. For more precise evaluation, add `expected_tool_calls` to your dataset.

Each expected tool call specifies:
- `name` — which tool should be called
- `arguments` (optional) — what arguments it should be called with

### Three matching modes

| Mode | Config | How it compares |
|---|---|---|
| **Fuzzy** (default) | `ToolCallCorrectness()` | LLM semantically compares actual vs expected |
| **Exact** | `ToolCallCorrectness(should_exact_match=True)` | String comparison of names and arguments |
| **Ordered** | `ToolCallCorrectness(should_exact_match=True, should_consider_ordering=True)` | Exact match + order matters |

For our cooking assistant, **fuzzy matching** is the right default — we care that the agent searched for "pancakes" but not whether it passed `"pancakes"` vs `"Pancakes"` vs `"classic pancakes"`.

In [ ]:
eval_dataset_with_tools = [
    # ── Should call search_recipes ────────────────────────────────────────
    {
        "inputs": {"question": "How do I make pancakes?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "pancakes"}},
            ],
        },
    },
    {
        "inputs": {"question": "I want to cook pasta carbonara tonight."},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "pasta carbonara"}},
            ],
        },
    },
    # ── Should call get_nutrition_info ─────────────────────────────────────
    {
        "inputs": {"question": "How many calories are in an avocado?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_nutrition_info", "arguments": {"food_item": "avocado"}},
            ],
        },
    },
    {
        "inputs": {"question": "What are the macros in chicken breast?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_nutrition_info", "arguments": {"food_item": "chicken breast"}},
            ],
        },
    },
    # ── Should NOT call any tools ─────────────────────────────────────────
    {
        "inputs": {"question": "What's the difference between baking and roasting?"},
        "expectations": {
            "expected_tool_calls": [],
        },
    },
    {
        "inputs": {"question": "How do I properly season a cast iron pan?"},
        "expectations": {
            "expected_tool_calls": [],
        },
    },
    # ── Should call BOTH tools ────────────────────────────────────────────
    {
        "inputs": {"question": "Give me a chicken stir fry recipe and the nutrition info for chicken breast."},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "chicken stir fry"}},
                {"name": "get_nutrition_info", "arguments": {"food_item": "chicken breast"}},
            ],
        },
    },
]

print(f"Eval set size: {len(eval_dataset_with_tools)} examples")
print(f"  - recipe lookups: 2")
print(f"  - nutrition lookups: 2")
print(f"  - no tools expected: 2")
print(f"  - multi-tool: 1")

In [ ]:
results_fuzzy = mlflow.genai.evaluate(
    data=eval_dataset_with_tools,
    predict_fn=recipe_agent,
    scorers=[
        ToolCallCorrectness(),   # fuzzy match (default)
        ToolCallEfficiency(),
    ],
)

print("=== Fuzzy matching with expected_tool_calls ===")
print(results_fuzzy.metrics)

In [ ]:
results_fuzzy.tables["eval_results_table"]

### Exact matching — when precision matters

Fuzzy matching is forgiving — `"pancakes"` matches `"Pancakes"` semantically. But sometimes you want strict validation, for example when testing that the agent passes structured IDs or coordinates.

In [ ]:
results_exact = mlflow.genai.evaluate(
    data=eval_dataset_with_tools,
    predict_fn=recipe_agent,
    scorers=[
        ToolCallCorrectness(should_exact_match=True),
    ],
)

print("=== Exact matching ===")
print(results_exact.metrics)

In [ ]:
# Compare fuzzy vs exact — which rows diverge?
# Rows that pass fuzzy but fail exact reveal where the model paraphrases arguments.
results_exact.tables["eval_results_table"]

---
## 8 — Combining Tool Call and Output Quality Scorers

Tool correctness alone doesn't tell you if the agent gave a *good answer*. The agent might call the right tool but then ignore the results, or summarize them poorly.

The real power is combining tool-call scorers with the output-quality scorers from Stage 1. This gives you a complete picture:

| Layer | What it checks | Scorers |
|---|---|---|
| **Actions** | Did the agent do the right things? | `ToolCallCorrectness`, `ToolCallEfficiency` |
| **Output quality** | Is the final answer good? | `Correctness`, `Guidelines`, `@scorer` |
| **Safety** | Is the answer safe? | `Safety` |

In [ ]:
import re
from mlflow.genai.scorers import (
    Correctness,
    Guidelines,
    RelevanceToQuery,
    Safety,
    ToolCallCorrectness,
    ToolCallEfficiency,
    scorer,
    Feedback,
)


@scorer
def has_measurement(outputs: str, **kwargs) -> bool:
    """Checks that the response includes at least one specific measurement."""
    pattern = re.compile(
        r'\b\d+\.?\d*\s*'
        r'(cup|tbsp|tsp|tablespoon|teaspoon|oz|ounce|lb|pound|g|gram|kg|'
        r'ml|litre|liter|°[CF]|degree|minute|min|hour|hr|calorie|kcal)s?\b',
        re.IGNORECASE,
    )
    return bool(pattern.search(outputs))


# Full eval set with both tool expectations AND content expectations
eval_dataset_full = [
    {
        "inputs": {"question": "How do I make pancakes?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "pancakes"}},
            ],
            "expected_facts": ["flour", "baking powder", "milk", "egg", "butter"],
        },
    },
    {
        "inputs": {"question": "How many calories are in an avocado?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_nutrition_info", "arguments": {"food_item": "avocado"}},
            ],
            "expected_facts": ["240", "calories"],
        },
    },
    {
        "inputs": {"question": "What's the difference between baking and roasting?"},
        "expectations": {
            "expected_tool_calls": [],
            "expected_response": (
                "Both use dry heat in an oven. Baking is typically for doughs and batters "
                "(bread, cakes, pastries) at moderate temperatures. Roasting is for meats "
                "and vegetables at higher temperatures to develop browning and caramelization."
            ),
        },
    },
    {
        "inputs": {"question": "I want to cook pasta carbonara tonight."},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "pasta carbonara"}},
            ],
            "expected_facts": ["guanciale", "egg yolks", "Pecorino", "pasta water"],
        },
    },
    {
        "inputs": {"question": "Give me a chicken stir fry recipe and the nutrition info for chicken breast."},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "chicken stir fry"}},
                {"name": "get_nutrition_info", "arguments": {"food_item": "chicken breast"}},
            ],
            "expected_facts": ["soy sauce", "165 calories", "protein"],
        },
    },
]

print(f"Full eval set: {len(eval_dataset_full)} examples")

In [ ]:
results_full = mlflow.genai.evaluate(
    data=eval_dataset_full,
    predict_fn=recipe_agent,
    scorers=[
        # ── Action quality ────────────────────────────────────────────────
        ToolCallCorrectness(),
        ToolCallEfficiency(),
        # ── Output quality ────────────────────────────────────────────────
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="uses_tool_results",
            guidelines=(
                "When the agent has called a tool and received results, the final response "
                "must incorporate the data from the tool results rather than ignoring them "
                "or making up different information."
            ),
        ),
        has_measurement,
        # ── Safety ────────────────────────────────────────────────────────
        Safety(),
    ],
)

print("=== Full evaluation: tool calls + output quality ===")
print(results_full.metrics)

In [ ]:
results_full.tables["eval_results_table"]

---
## 9 — Common Tool-Calling Failures

Let's deliberately test some edge cases that tool-calling agents often get wrong. These are the kinds of failures you want your eval set to catch *before* users find them.

| Failure mode | What happens | How to catch it |
|---|---|---|
| **Tool when none needed** | Agent calls `search_recipes` for a general knowledge question | `expected_tool_calls: []` |
| **Wrong tool** | Agent calls `get_nutrition_info` when asked for a recipe | `expected_tool_calls` with correct tool name |
| **Redundant calls** | Agent calls `search_recipes("pancakes")` twice | `ToolCallEfficiency` |
| **Missing tool call** | Agent answers from memory instead of looking up the recipe | `ToolCallCorrectness` with expectations |
| **Bad arguments** | Agent passes `"food"` instead of `"chicken breast"` | `expected_tool_calls` with arguments |

In [ ]:
edge_case_dataset = [
    # Agent should NOT use tools for a general knowledge question
    {
        "inputs": {"question": "What temperature should my oven be for baking bread?"},
        "expectations": {
            "expected_tool_calls": [],
        },
    },
    # Agent should call search_recipes, not answer from memory
    {
        "inputs": {"question": "What's in your chicken stir fry recipe?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes", "arguments": {"query": "chicken stir fry"}},
            ],
        },
    },
    # Ambiguous — could be nutrition or recipe. Either tool is acceptable.
    {
        "inputs": {"question": "Tell me about eggs."},
        "expectations": {},
    },
    # Should use nutrition tool with a specific item, not a vague query
    {
        "inputs": {"question": "Is spaghetti high in carbs?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_nutrition_info", "arguments": {"food_item": "spaghetti"}},
            ],
        },
    },
]

results_edge = mlflow.genai.evaluate(
    data=edge_case_dataset,
    predict_fn=recipe_agent,
    scorers=[
        ToolCallCorrectness(),
        ToolCallEfficiency(),
    ],
)

print("=== Edge case evaluation ===")
print(results_edge.metrics)

In [ ]:
results_edge.tables["eval_results_table"]

---
## 10 — Partial Expectations: Tool Names Only

Sometimes you care *which* tool was called but not the exact arguments. This is useful for:
- Early-stage eval when you're still iterating on argument formats
- Cases where multiple argument variations are all acceptable

Just omit the `arguments` key from `expected_tool_calls`.

In [ ]:
names_only_dataset = [
    {
        "inputs": {"question": "Find me a pasta recipe."},
        "expectations": {
            "expected_tool_calls": [
                {"name": "search_recipes"},
            ],
        },
    },
    {
        "inputs": {"question": "How healthy is an egg?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_nutrition_info"},
            ],
        },
    },
]

results_names = mlflow.genai.evaluate(
    data=names_only_dataset,
    predict_fn=recipe_agent,
    scorers=[ToolCallCorrectness(should_exact_match=True)],
)

print("=== Names-only matching ===")
print(results_names.metrics)

In [ ]:
results_names.tables["eval_results_table"]

---
## 11 — Comparing Runs in the MLflow UI

Open the MLflow UI and navigate to the `recipe-assistant-tools` experiment. You should see multiple evaluation runs.

**What to compare:**

- **Ground-truth-free vs fuzzy matching:** Does adding `expected_tool_calls` change the scores? If the ground-truth-free mode already catches most issues, you may not need labeled data for every row.
- **Fuzzy vs exact matching:** Which rows pass fuzzy but fail exact? Those show where the model paraphrases arguments — decide whether that matters for your use case.
- **Tool correctness vs output quality:** A row can have perfect tool calls but a poor final answer (or vice versa). The combined eval reveals these gaps.

Click into individual traces to see the full span tree — this is where you'll understand *why* a tool call was scored as incorrect.

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| `@mlflow.trace(span_type=SpanType.TOOL)` | Made tool calls visible as `TOOL` spans in traces |
| `@mlflow.trace` on the agent | Captured the full agent loop as a single trace with nested spans |
| OpenAI function calling format | Described tools to the LLM with JSON schema |
| Tool-calling agent loop | Send → check for tool calls → execute → send again |
| `ToolCallCorrectness()` | Evaluated whether the agent called the right tools |
| `ToolCallEfficiency()` | Checked for redundant or unnecessary tool calls |
| `expected_tool_calls` | Ground truth for which tools should be called with which arguments |
| Fuzzy vs exact matching | Two modes for comparing actual vs expected tool calls |
| Names-only expectations | Partial ground truth — check tool selection without argument matching |
| Combined eval suite | Tool-call scorers + output-quality scorers in one evaluation run |

### The key insight

Evaluating an agent is not just about evaluating its *answers* — it's about evaluating its *actions*. A tool-calling agent can give the right answer for the wrong reasons (hallucinating instead of looking it up) or take the right actions but produce a poor summary. You need both layers.

```
Action quality:   Did it call the right tools? → ToolCallCorrectness
                  Did it call them efficiently? → ToolCallEfficiency
Output quality:   Is the answer correct?        → Correctness
                  Did it use the tool results?   → Guidelines
Safety:           Is it safe?                    → Safety
```

---

**Next → Stage 3:** We'll explore how to iterate on the agent when tool-call evaluations reveal problems — prompt engineering for better tool selection, handling tool errors gracefully, and building a regression suite that catches tool-calling regressions across model updates.